In [1]:
import pandas as pd
from pyfaidx import Fasta
import numpy as np
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq
import gc

# ================= 1. CẤU HÌNH HỆ THỐNG =================
GTF_FILE = r"C:\Users\dotru\STUDIE\FPTU\AiTA_Lab\gencode.v49.basic.annotation.gtf"
FASTA_FILE = r"C:\Users\dotru\STUDIE\FPTU\AiTA_Lab\Homo_sapiens.GRCh38.dna.primary_assembly.fa"
OUTPUT_FILE = r'C:\Users\dotru\STUDIE\FPTU\AiTA_Lab\data\ver2\dataset3\pre_train_splicing_prediction.parquet' 

WINDOW = 300                
SEQ_LEN = (WINDOW * 2) + 1  
WRITE_BATCH_SIZE = 100000   
NEGATIVE_MULTIPLIER = 500
VALID_CHROMS = set([str(i) for i in range(1, 23) if i not in [20, 21]] + ['X', 'Y', 'M', 'MT'])

TRANS_TABLE = str.maketrans("ATCG", "TAGC")

# ================= 2. CÁC HÀM XỬ LÝ =================
class GenomeBitTracker:
    def __init__(self, genome_keys, genome_dict):
        print("      -> Khởi tạo Bitmask Map (~800MB RAM)...")
        self.bitmaps = {}
        for chrom in VALID_CHROMS:
            fasta_key = chrom if chrom in genome_keys else f"chr{chrom}" if f"chr{chrom}" in genome_keys else None
            if fasta_key:
                length_bp = len(genome_dict[fasta_key]) + 20000
                size_in_bytes = (length_bp // 8) + 1
                self.bitmaps[(chrom, '+')] = np.zeros(size_in_bytes, dtype=np.uint8)
                self.bitmaps[(chrom, '-')] = np.zeros(size_in_bytes, dtype=np.uint8)

    def is_seen(self, chrom, pos, strand):
        if pos < 0: return True
        try: return (self.bitmaps[(chrom, strand)][pos // 8] & (1 << (pos % 8))) != 0
        except: return True

    def mark_seen(self, chrom, pos, strand):
        if pos >= 0:
            try: self.bitmaps[(chrom, strand)][pos // 8] |= (1 << (pos % 8))
            except: pass

def normalize_chrom(chrom_name):
    s = str(chrom_name).strip()
    if s.startswith("NC_"):
        try:
            val = int(s.split('_')[1].split('.')[0])
            if val == 23: return 'X'
            if val == 24: return 'Y'
            if val == 12920: return 'M'
            return str(val)
        except: return s
    if s.lower().startswith('chr'):
        raw = s[3:]
        if raw == 'M': return 'M'
        return raw
    return s

# ================= 3. CHƯƠNG TRÌNH CHÍNH =================
print(f"--- GENERATING MASSIVE DATASET (GOD-TIER SPEED V3) ---")

print("[1/4] Loading Genome (Fasta -> RAM & Numpy)...")
lazy_genome = Fasta(FASTA_FILE, sequence_always_upper=True)

genome_ram = {}
genome_ram_np = {} # Bản sao Byte Array cho Numpy
for key in lazy_genome.keys():
    seq_str = str(lazy_genome[key])
    genome_ram[key] = seq_str
    genome_ram_np[key] = np.frombuffer(seq_str.encode('ascii'), dtype=np.uint8)

tracker = GenomeBitTracker(lazy_genome.keys(), genome_ram)

print("[2/4] Parsing GTF & Building Data...")
df = pd.read_csv(GTF_FILE, sep='\t', comment='#', header=None, usecols=[0, 2, 3, 4, 6, 8],
                    names=['chrom', 'feature', 'start', 'end', 'strand', 'attribute'])
df = df[(df['feature'] == 'exon') & (df['attribute'].str.contains('gene_type "protein_coding"'))]

df['norm_chrom'] = df['chrom'].apply(normalize_chrom)
df = df[df['norm_chrom'].isin(VALID_CHROMS)]

def get_fkey(c):
    if c in genome_ram: return c
    if f"chr{c}" in genome_ram: return f"chr{c}"
    if c == 'M' and 'chrM' in genome_ram: return 'chrM'
    return None

df['fasta_key'] = df['norm_chrom'].apply(get_fkey)
df = df.dropna(subset=['fasta_key'])

print(f"      -> {len(df)} valid exons ready.")

schema = pa.schema([('id', pa.string()), ('dna', pa.string()), ('label', pa.int8())])
def flush_buffer_to_parquet(writer, buffer):
    if not buffer: return
    table = pa.Table.from_pandas(pd.DataFrame(buffer, columns=['id', 'dna', 'label']), schema=schema)
    writer.write_table(table)

with pq.ParquetWriter(OUTPUT_FILE, schema, compression='snappy') as writer:
    
    # ================= BƯỚC 3: XỬ LÝ POSITIVE =================
    print("\n[3/4] Processing Positives...")
    pos_buffer = []    
    total_positives = 0
    
    for row in tqdm(df.itertuples(index=False), total=len(df), desc="Positives"):
        nc, fk, st = row.norm_chrom, row.fasta_key, row.strand
        donor_pos = row.end + 1 if st == '+' else row.start - 1
        acc_pos = row.start - 1 if st == '+' else row.end + 1

        for pos, lbl, motif_check in [(donor_pos, 1, ['GT', 'GC']), (acc_pos, 2, ['AG'])]:
            if not tracker.is_seen(nc, pos, st):
                start_idx, end_idx = pos - 1 - WINDOW, pos + WINDOW
                if start_idx >= 0 and end_idx < len(genome_ram[fk]):
                    seq = genome_ram[fk][start_idx:end_idx]
                    if len(seq) == SEQ_LEN and 'N' not in seq:
                        if st == '-': seq = seq.translate(TRANS_TABLE)[::-1]
                        motif = seq[WINDOW:WINDOW+2] if lbl == 1 else seq[WINDOW-1:WINDOW+1]
                        
                        if motif in motif_check or (lbl==2 and motif=='AG'):
                            pos_buffer.append([f"{'Donor' if lbl==1 else 'Acc'}_{nc}_{pos}_{st}", seq, lbl])
                            tracker.mark_seen(nc, pos, st)
                            total_positives += 1
        
        if len(pos_buffer) >= WRITE_BATCH_SIZE:
            flush_buffer_to_parquet(writer, pos_buffer)
            pos_buffer = [] 

    flush_buffer_to_parquet(writer, pos_buffer)
    pos_buffer = []
    print(f"      -> Đã ghi {total_positives} mẫu Positive sạch.")
    
    # ================= BƯỚC 4: TIỀN TÍNH TOÁN VỊ TRÍ NEGATIVE BẰNG NUMPY =================
    target_neg = total_positives * NEGATIVE_MULTIPLIER
    print(f"\n[4/4] Generating {target_neg:,} Negatives (Vectorized Exact Search)...")
    
    candidate_dict = {}
    total_candidates = 0
    map_nc_to_fk = {}

    print("      -> Quét toàn bộ Hệ gen bằng Numpy C-Core (Chỉ mất vài giây)...")
    for chrom in tqdm(VALID_CHROMS, desc="Scanning Chromosomes"):
        fk = df[df['norm_chrom'] == chrom]['fasta_key'].iloc[0] if len(df[df['norm_chrom'] == chrom]) > 0 else None
        if not fk: continue
        map_nc_to_fk[chrom] = fk
        
        arr = genome_ram_np[fk]
        arr_len = len(arr)
        
        # 1. Tạo Bitmask vùng lân cận Exon (+/- 10000)
        rm = np.zeros(arr_len, dtype=bool)
        chrom_df = df[df['norm_chrom'] == chrom]
        starts = np.maximum(WINDOW, chrom_df['start'].values - 10001)
        ends = np.minimum(arr_len - WINDOW, chrom_df['end'].values + 10000)
        
        for s, e in zip(starts, ends):
            if s < e: rm[s:e] = True
            
        # 2. Cắt mảng (tránh tràn viền khi check motif)
        rm_slice = rm[WINDOW : -WINDOW]
        c = arr[WINDOW : -WINDOW]
        p = arr[WINDOW-1 : -WINDOW-1]
        n = arr[WINDOW+1 : -WINDOW+1]
        
        # 3. Motif Regex siêu tốc bằng Vector Bool
        # A=65, C=67, G=71, T=84
        valid_pos = rm_slice & (c == 71) & ((n == 84) | (n == 67) | (p == 65))
        pos_1based = np.where(valid_pos)[0] + WINDOW + 1
        
        valid_neg = rm_slice & (c == 67) & ((p == 65) | (p == 71) | (n == 84))
        neg_1based = np.where(valid_neg)[0] + WINDOW + 1
        
        candidate_dict[chrom] = {'+': pos_1based, '-': neg_1based}
        total_candidates += len(pos_1based) + len(neg_1based)
        
    print(f"      -> TÌM THẤY {total_candidates:,} VỊ TRÍ HỢP LỆ TRONG VŨ TRỤ GEN!")
    
    # Dọn rác RAM cực mạnh
    del genome_ram_np
    gc.collect()
    
    # ================= BƯỚC 5: LẤY MẪU NEGATIVE VÀ GHI ĐĨA =================
    target_neg = min(target_neg, total_candidates) # Bảo vệ nếu chỉ tiêu lớn hơn vũ trụ gen
    
    neg_count = 0
    neg_buffer = []
    pbar = tqdm(total=target_neg, desc="Mining Negatives")
    
    for chrom, strands in candidate_dict.items():
        fk = map_nc_to_fk[chrom]
        for st, arr in strands.items():
            if len(arr) == 0: continue
            
            # Chia quota phân bổ đều cho từng NST và Mạch
            quota = int(target_neg * (len(arr) / total_candidates))
            
            # Trộn ngẫu nhiên tọa độ
            np.random.shuffle(arr)
            
            st_count = 0
            for pos in arr:
                if st_count >= quota and neg_count >= target_neg: break
                
                if tracker.is_seen(chrom, pos, st): continue
                    
                start_idx = pos - 1 - WINDOW
                end_idx = pos + WINDOW
                seq = genome_ram[fk][start_idx:end_idx]
                if 'N' in seq: continue # Loại bỏ chuỗi kém chất lượng
                    
                if st == '-': 
                    seq = seq.translate(TRANS_TABLE)[::-1]
                    
                neg_buffer.append([f"Neg_{chrom}_{pos}_{st}", seq, 0])
                tracker.mark_seen(chrom, pos, st)
                
                st_count += 1
                neg_count += 1
                pbar.update(1)
                
                if len(neg_buffer) >= WRITE_BATCH_SIZE:
                    flush_buffer_to_parquet(writer, neg_buffer)
                    neg_buffer = []

    if neg_buffer: flush_buffer_to_parquet(writer, neg_buffer)
        
print(f"\n✅ HOÀN TẤT VỚI TỐC ĐỘ ÁNH SÁNG! File saved at: {OUTPUT_FILE}")
print(f"      Tổng Negatives thực tế: {neg_count:,}")

--- GENERATING MASSIVE DATASET (GOD-TIER SPEED V3) ---
[1/4] Loading Genome (Fasta -> RAM & Numpy)...
      -> Khởi tạo Bitmask Map (~800MB RAM)...
[2/4] Parsing GTF & Building Data...
      -> 2163595 valid exons ready.

[3/4] Processing Positives...


Positives: 100%|██████████| 2163595/2163595 [00:23<00:00, 91554.10it/s] 


      -> Đã ghi 479067 mẫu Positive sạch.

[4/4] Generating 239,533,500 Negatives (Vectorized Exact Search)...
      -> Quét toàn bộ Hệ gen bằng Numpy C-Core (Chỉ mất vài giây)...


Scanning Chromosomes: 100%|██████████| 24/24 [00:29<00:00,  1.25s/it]


      -> TÌM THẤY 288,212,888 VỊ TRÍ HỢP LỆ TRONG VŨ TRỤ GEN!


Mining Negatives: 279634403it [42:47, 77157.14it/s]                                


✅ HOÀN TẤT VỚI TỐC ĐỘ ÁNH SÁNG! File saved at: C:\Users\dotru\STUDIE\FPTU\AiTA_Lab\data\ver2\dataset3\pre_train_splicing_prediction.parquet
      Tổng Negatives thực tế: 279,643,178


In [1]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
from tqdm import tqdm

# --- CẤU HÌNH ---
TARGET_FILE = r'D:\\pre_train_splicing_prediction.parquet'
TEMP_FILE = TARGET_FILE + '.tmp.parquet'
NEW_CHROM_COL = 'CHROM'
BATCH_SIZE = 100000  # Đọc từng 100k dòng để tiết kiệm RAM

# Map đổi tên: {'Tên cũ': 'Tên mới'}
RENAME_MAP = {
    'label': 'Splicing_types',
    'dna': 'sequence'
}

print(f"🔄 Đang xử lý file: {TARGET_FILE}")
print(f"   - Đổi tên cột: {RENAME_MAP}")
print(f"   - Thêm/Update cột: {NEW_CHROM_COL}")

try:
    if not os.path.exists(TARGET_FILE):
        raise FileNotFoundError(f"Không tìm thấy file '{TARGET_FILE}'")

    # Mở file Parquet để đọc meta-data
    parquet_file = pq.ParquetFile(TARGET_FILE)
    total_rows = parquet_file.metadata.num_rows
    total_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"📊 Tổng số dòng: {total_rows:,} | Số lô (batches): {total_batches:,}")

    writer = None  # Sẽ khởi tạo sau khi xác định được Schema mới

    # Iter_batches giúp đọc streaming qua Parquet mà không nổ RAM
    for batch in tqdm(parquet_file.iter_batches(batch_size=BATCH_SIZE), total=total_batches, desc="Processing Batches"):
        
        # Chuyển batch (PyArrow) sang Pandas để dễ xử lý logic
        df = batch.to_pandas()
        
        # 1. Đổi tên cột
        df.rename(columns=RENAME_MAP, inplace=True)
        
        # 2. Xử lý Logic lấy Chromosome (Tốc độ siêu cao nhờ Vectorization của Pandas)
        # Tách id thành các phần, ví dụ "Donor_1_65434_+" -> lấy phần tử [1] là "1"
        chrom_series = df['id'].astype(str).str.split('_').str[1]
        
        # Điền "unknown" nếu có dòng bị lỗi cấu trúc id
        chrom_series = chrom_series.fillna("unknown")
        
        # Thêm tiền tố 'chr' nếu chưa có và không phải là 'unknown'
        needs_prefix_mask = (~chrom_series.str.startswith('chr')) & (chrom_series != 'unknown')
        df[NEW_CHROM_COL] = np.where(needs_prefix_mask, 'chr' + chrom_series, chrom_series)
        
        # 3. Chuyển lại thành PyArrow Table
        table = pa.Table.from_pandas(df)
        
        # 4. Khởi tạo Writer (Chỉ chạy ở vòng lặp đầu tiên khi đã có Schema mới)
        if writer is None:
            # Dùng chuẩn nén snappy giống code crawl data lúc trước
            writer = pq.ParquetWriter(TEMP_FILE, table.schema, compression='snappy')
            
        # Ghi batch vào đĩa
        writer.write_table(table)

    # Đóng Writer (file tạm)
    if writer:
        writer.close()

    # [BẢN VÁ QUAN TRỌNG] Giải phóng file Reader gốc
    del parquet_file
    import gc
    gc.collect()

    # Bây giờ Windows đã cho phép thay thế file cũ thoải mái
    if os.path.exists(TARGET_FILE):
        os.remove(TARGET_FILE) 
    os.rename(TEMP_FILE, TARGET_FILE)
    
    print(f"\n✅ HOÀN TẤT! File đã được cập nhật thành công.")

except Exception as e:
    # Fallback: dọn dẹp file tạm nếu bị lỗi giữa chừng
    if os.path.exists(TEMP_FILE):
        os.remove(TEMP_FILE)
    print(f"\n❌ Lỗi Runtime: {e}")

🔄 Đang xử lý file: D:\\pre_train_splicing_prediction.parquet
   - Đổi tên cột: {'label': 'Splicing_types', 'dna': 'sequence'}
   - Thêm/Update cột: CHROM
📊 Tổng số dòng: 280,122,245 | Số lô (batches): 2,802


Processing Batches: 100%|██████████| 2802/2802 [21:08<00:00,  2.21it/s]



✅ HOÀN TẤT! File đã được cập nhật thành công.


In [2]:
import os
import time
from collections import Counter
import pandas as pd
import pyarrow.parquet as pq
from tqdm import tqdm

# ================= CẤU HÌNH =================
# Đường dẫn đã được đổi sang .parquet
FILE_PATH = r'D:\\pre_train_splicing_prediction.parquet'
SAMPLE_ROWS = 5  # Số dòng in mẫu
BATCH_SIZE = 250000  # Đọc 250k dòng mỗi lô để quét cực nhanh

def inspect_and_validate_full(file_path):
    print(f"🚀 BẮT ĐẦU KIỂM TRA FILE PARQUET: {file_path}")
    
    start_time = time.time()
    
    # 1. BIẾN THỐNG KÊ
    stats = {
        "total_rows": 0,
        "null_counts": Counter(),
        "seq_lengths": Counter(),      # Thống kê độ dài sequence
        "label_counts": Counter(),     # Thống kê Splicing_types
        "chrom_counts": Counter(),     # Thống kê CHROM
        "samples": []
    }
    
    try:
        if not os.path.exists(file_path):
            raise FileNotFoundError
            
        # Mở file Parquet (chỉ đọc Meta-data ban đầu, chưa load vào RAM)
        parquet_file = pq.ParquetFile(file_path)
        total_rows_meta = parquet_file.metadata.num_rows
        header = parquet_file.schema.names
        
        # Tìm TÊN CỘT dựa trên các keyword
        col_id = next((c for c in header if c in ['id', 'ID']), None)
        col_seq = next((c for c in header if c in ['sequence', 'dna']), None)
        col_lbl = next((c for c in header if c in ['Splicing_types', 'label']), None)
        col_chr = next((c for c in header if c in ['CHROM', 'chrom']), None)
        
        # Khởi tạo đếm null cho tất cả các cột
        for col in header: 
            stats["null_counts"][col] = 0
            
        print(f"   -> Header tìm thấy: {header}")
        print(f"   -> Mapping columns: ID={col_id}, Seq={col_seq}, Label={col_lbl}, Chrom={col_chr}")
        print(f"   -> Tổng số dòng (từ Metadata): {total_rows_meta:,}")

        # --- 2. QUÉT DỮ LIỆU (STREAMING THEO BATCH) ---
        total_batches = (total_rows_meta + BATCH_SIZE - 1) // BATCH_SIZE
        pbar = tqdm(parquet_file.iter_batches(batch_size=BATCH_SIZE), total=total_batches, desc="Analyzing Batches")
        
        for batch in pbar:
            # Chuyển batch sang Pandas DataFrame để tận dụng tốc độ tính toán C
            df = batch.to_pandas()
            stats["total_rows"] += len(df)
            
            # A. LẤY MẪU (Chỉ lấy ở batch đầu tiên)
            if len(stats["samples"]) < SAMPLE_ROWS:
                needed = SAMPLE_ROWS - len(stats["samples"])
                # Lấy số dòng cần thiết và chuyển thành list of dicts
                samples_df = df.head(needed)
                stats["samples"].extend(samples_df.to_dict('records'))
            
            # B. CHECK NULL & VALIDATE (Nhanh hơn rất nhiều với Vectorization)
            for col in header:
                # Tính tổng các giá trị NA/Null
                null_count = df[col].isna().sum()
                # Nếu là cột chuỗi (string), tính thêm các chuỗi rỗng ""
                if df[col].dtype == object or pd.api.types.is_string_dtype(df[col]):
                    null_count += (df[col].astype(str).str.strip() == '').sum()
                stats["null_counts"][col] += int(null_count)
            
            # C. THỐNG KÊ GIÁ TRỊ
            if col_seq in df.columns:
                # Nhóm theo độ dài chuỗi và cộng dồn
                seq_lens = df[col_seq].str.len().value_counts().to_dict()
                for length, count in seq_lens.items():
                    stats["seq_lengths"][length] += count
                    
            if col_lbl in df.columns:
                lbls = df[col_lbl].value_counts().to_dict()
                for lbl, count in lbls.items():
                    stats["label_counts"][lbl] += count
                    
            if col_chr in df.columns:
                chrs = df[col_chr].value_counts().to_dict()
                for ch, count in chrs.items():
                    stats["chrom_counts"][ch] += count

    except FileNotFoundError:
        print(f"❌ Lỗi: Không tìm thấy file {file_path}")
        return
    except Exception as e:
        print(f"❌ Lỗi cấu trúc Parquet: {e}")
        return

    # --- 3. BÁO CÁO KẾT QUẢ ---
    elapsed = time.time() - start_time
    total = stats["total_rows"]
    
    print("\n" + "="*80)
    print(f"📊 BÁO CÁO DỮ LIỆU (Time: {elapsed:.2f}s | Total: {total:,} rows)")
    print("="*80)

    # 3.1 Bảng mẫu
    print(f"\n1. MẪU DỮ LIỆU ({SAMPLE_ROWS} dòng đầu):")
    print(f"   {'ID':<25} | {'Sequence (Preview)':<20} | {'Type':<10} | {'Chrom':<8}")
    print(f"   {'-'*25} | {'-'*20} | {'-'*10} | {'-'*8}")
    
    for row in stats["samples"]:
        _id = str(row.get(col_id, "N/A")) if col_id else "N/A"
        _seq = str(row.get(col_seq, "N/A")) if col_seq else "N/A"
        _lbl = str(row.get(col_lbl, "N/A")) if col_lbl else "N/A"
        _chr = str(row.get(col_chr, "N/A")) if col_chr else "N/A"
        
        seq_show = _seq[:6] + "..." + _seq[-4:] if len(_seq) > 10 else _seq
        print(f"   {_id:<25} | {seq_show:<20} | {_lbl:<10} | {_chr:<8}")

    # 3.2 Báo cáo Null
    print(f"\n2. KIỂM TRA NULL (COMPLETENESS):")
    has_null = False
    print(f"   {'Tên Cột':<20} | {'Số Null':<10} | {'% Lỗi'}")
    for col, count in stats["null_counts"].items():
        pct = (count / total * 100) if total > 0 else 0
        status = "✅" if count == 0 else f"❌ {count:,}"
        print(f"   {col:<20} | {status:<10} | {pct:.4f}%")
        if count > 0: has_null = True
    if not has_null: print("   -> Sạch 100%.")

    # 3.3 Phân bố Label
    print(f"\n3. PHÂN BỐ TYPE (Splicing_types):")
    for lbl, count in stats["label_counts"].items():
        print(f"   - '{lbl}': {count:,} mẫu")

    # 3.4 Phân bố Chrom
    print(f"\n4. PHÂN BỐ CHROMOSOME (Top 5):")
    for ch, count in stats["chrom_counts"].most_common(5):
        print(f"   - {ch}: {count:,} mẫu")
    if len(stats["chrom_counts"]) > 5: print(f"   ... và {len(stats['chrom_counts'])-5} chrom khác.")

    # 3.5 Sequence Length
    print(f"\n5. SEQUENCE LENGTH:")
    for length, count in stats["seq_lengths"].items():
        # Kiểm tra xem chiều dài có phải 601 (WINDOW 300 * 2 + 1) hoặc null không
        if pd.isna(length): continue
        status = "✅ Chuẩn" if length == 601 else "⚠️ LỆCH"
        print(f"   - Length {int(length)}: {count:,} mẫu ({status})")

inspect_and_validate_full(FILE_PATH)

🚀 BẮT ĐẦU KIỂM TRA FILE PARQUET: D:\\pre_train_splicing_prediction.parquet
   -> Header tìm thấy: ['id', 'sequence', 'Splicing_types', 'CHROM']
   -> Mapping columns: ID=id, Seq=sequence, Label=Splicing_types, Chrom=CHROM
   -> Tổng số dòng (từ Metadata): 280,122,245


Analyzing Batches:   0%|          | 0/1121 [00:00<?, ?it/s]

Analyzing Batches: 100%|██████████| 1121/1121 [09:38<00:00,  1.94it/s]


📊 BÁO CÁO DỮ LIỆU (Time: 578.19s | Total: 280,122,245 rows)

1. MẪU DỮ LIỆU (5 dòng đầu):
   ID                        | Sequence (Preview)   | Type       | Chrom   
   ------------------------- | -------------------- | ---------- | --------
   Donor_1_65434_+           | AAAAGT...TTAA        | 1          | chr1    
   Donor_1_65574_+           | GATAGC...TTCC        | 1          | chr1    
   Acc_1_65519_+             | CTTTAT...CTCC        | 2          | chr1    
   Acc_1_69036_+             | AAAGGA...GCGC        | 2          | chr1    
   Donor_1_924949_+          | ACCTCA...GAGC        | 1          | chr1    

2. KIỂM TRA NULL (COMPLETENESS):
   Tên Cột              | Số Null    | % Lỗi
   id                   | ✅          | 0.0000%
   sequence             | ✅          | 0.0000%
   Splicing_types       | ✅          | 0.0000%
   CHROM                | ✅          | 0.0000%
   -> Sạch 100%.

3. PHÂN BỐ TYPE (Splicing_types):
   - '1': 241,995 mẫu
   - '2': 237,072 mẫu
   - '0': 279,

In [3]:
import os
import time
from collections import Counter
import pandas as pd
import pyarrow.parquet as pq
from tqdm import tqdm

# ================= CẤU HÌNH =================
# Đường dẫn tới file Parquet của bạn
FILE_PATH = r'D:\\pre_train_splicing_prediction.parquet'
WINDOW = 300 
CENTER_IDX = WINDOW  # Tâm nằm ở index 300
BATCH_SIZE = 250000  # Đọc 250k dòng/lần để siêu tối ưu RAM

def validate_biological_data(file_path):
    print(f"🧬 BẮT ĐẦU KIỂM TRA LOGIC SINH HỌC (PARQUET STREAMING): {file_path}")
    
    start_time = time.time()
    
    stats = {
        "total": 0,
        "invalid_chars": 0,   # Sequence chứa ký tự lạ (khác A, C, G, T, N)
        "motif_errors": 0,    # Sai Motif (Donor k phải GT/GC, Acc k phải AG)
        "wrong_length": 0,    # Số sequence không đủ độ dài 601bp
        "label_dist": Counter(),
        "chrom_dist": Counter()
    }
    
    try:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

        # Đọc Metadata để lấy cấu trúc cột
        parquet_file = pq.ParquetFile(file_path)
        total_rows_meta = parquet_file.metadata.num_rows
        header = parquet_file.schema.names
        
        col_seq = next((c for c in header if c in ['sequence', 'dna']), None)
        col_lbl = next((c for c in header if c in ['Splicing_types', 'label']), None)
        col_chr = next((c for c in header if c in ['CHROM', 'chrom']), None)
        
        if not all([col_seq, col_lbl, col_chr]):
            print(f"❌ Lỗi: Không tìm thấy đủ các cột sequence, label, CHROM trong header: {header}")
            return
            
        print(f"   -> Mapping OK: Seq={col_seq}, Label={col_lbl}, Chrom={col_chr}")
        print(f"   -> Tổng số dòng (từ Metadata): {total_rows_meta:,}")

        # --- QUÉT DỮ LIỆU (THEO LÔ BATCHING) ---
        total_batches = (total_rows_meta + BATCH_SIZE - 1) // BATCH_SIZE
        
        for batch in tqdm(parquet_file.iter_batches(batch_size=BATCH_SIZE), total=total_batches, desc="Bio-Checking"):
            df = batch.to_pandas()
            stats["total"] += len(df)
            
            # Cập nhật phân bố Label & Chromosome
            stats["label_dist"].update(df[col_lbl].value_counts().to_dict())
            stats["chrom_dist"].update(df[col_chr].value_counts().to_dict())
            
            # Lọc bỏ giá trị Null trước khi xử lý chuỗi
            valid_seq_mask = df[col_seq].notna()
            df_valid = df[valid_seq_mask].copy()
            
            if df_valid.empty:
                continue

            # 1. CHECK KÝ TỰ LẠ (Vectorized Regex - An toàn với Null và Hoa/Thường)
            # Tìm các chuỗi chứa ký tự KHÔNG PHẢI là A, C, G, T, N (case=False: tính cả a,c,g,t,n)
            invalid_mask = df_valid[col_seq].str.contains(r'[^ACGTN]', case=False, regex=True, na=False)
            stats["invalid_chars"] += int(invalid_mask.sum())

            # 2. CHECK ĐỘ DÀI CHUỖI
            correct_len_mask = df_valid[col_seq].str.len() == (WINDOW * 2 + 1)
            stats["wrong_length"] += int((~correct_len_mask).sum())  # Đếm các chuỗi bị cắt sai độ dài
            
            # Chỉ check Motif trên các sequence có độ dài chuẩn 601
            df_correct_len = df_valid[correct_len_mask]
            
            if df_correct_len.empty:
                continue
                
            # Ép label về dạng chuỗi để so sánh an toàn
            lbls_str = df_correct_len[col_lbl].astype(str)
            seqs_str = df_correct_len[col_seq]
            
            # -- Lỗi Motif Donor (Label 1) --
            donor_mask = lbls_str == '1'
            if donor_mask.any():
                # Dùng str.slice() lấy 2 ký tự ở tâm (index 300, 301)
                donor_motifs = seqs_str[donor_mask].str.slice(CENTER_IDX, CENTER_IDX + 2).str.upper()
                bad_donors = (~donor_motifs.isin(['GT', 'GC'])).sum()
                stats["motif_errors"] += int(bad_donors)
                
            # -- Lỗi Motif Acceptor (Label 2) --
            acc_mask = lbls_str == '2'
            if acc_mask.any():
                # Dùng str.slice() lấy 2 ký tự sát tâm (index 299, 300)
                acc_motifs = seqs_str[acc_mask].str.slice(CENTER_IDX - 1, CENTER_IDX + 1).str.upper()
                bad_accs = (acc_motifs != 'AG').sum()
                stats["motif_errors"] += int(bad_accs)

    except Exception as e:
        print(f"\n❌ Lỗi Runtime: {e}")
        return

    # --- BÁO CÁO KẾT QUẢ ---
    elapsed = time.time() - start_time
    total = stats["total"]
    
    print("\n" + "="*65)
    print(f"🔬 KẾT QUẢ KIỂM TRA SINH HỌC (Time: {elapsed:.2f}s | {total:,} mẫu)")
    print("="*65)
    
    # 1. Ký tự lạ
    status_char = "✅ Sạch" if stats["invalid_chars"] == 0 else f"❌ {stats['invalid_chars']:,} mẫu lỗi"
    print(f"1. TỪ VỰNG DNA (Chỉ A,C,G,T,N): {status_char}")
    
    # 2. Sequence Length
    status_len = "✅ 100% chuẩn (601bp)" if stats["wrong_length"] == 0 else f"❌ {stats['wrong_length']:,} mẫu sai độ dài"
    print(f"2. SEQUENCE LENGTH:            {status_len}")
    
    # 3. Motif
    pct_motif = (stats["motif_errors"] / total * 100) if total > 0 else 0
    status_motif = "✅ Chuẩn xác" if pct_motif < 1.0 else f"⚠️ ĐÁNG NGỜ ({stats['motif_errors']:,} mẫu sai motif)"
    print(f"3. MOTIF (GT/GC cho Donor, AG cho Acc): {status_motif}")
    if stats["motif_errors"] > 0:
        print("   -> (Nên kiểm tra lại logic Slicing ở file sinh dữ liệu!)")

    # 4. Phân bố Label
    print(f"\n4. PHÂN BỐ LABEL (Splicing_types):")
    for lbl, count in stats["label_dist"].items():
        print(f"   - Label '{lbl}': {count:,} mẫu")

    # 5. Chromosome Check
    print(f"\n5. CHROMOSOME COVERAGE:")
    print(f"   - Tìm thấy {len(stats['chrom_dist'])} nhiễm sắc thể.")
    
    chrom_keys = [str(k) for k in stats['chrom_dist'].keys()]
    if not any('1' in k for k in chrom_keys):
        print("   ⚠️ CẢNH BÁO: Không thấy dữ liệu từ chr1!")
        
    print(f"   - Top 5 Chromosomes: ", end="")
    top5 = [f"{k}: {v:,}" for k, v in stats["chrom_dist"].most_common(5)]
    print(" | ".join(top5))

validate_biological_data(FILE_PATH)

🧬 BẮT ĐẦU KIỂM TRA LOGIC SINH HỌC (PARQUET STREAMING): D:\\pre_train_splicing_prediction.parquet
   -> Mapping OK: Seq=sequence, Label=Splicing_types, Chrom=CHROM
   -> Tổng số dòng (từ Metadata): 280,122,245


Bio-Checking:   0%|          | 0/1121 [00:00<?, ?it/s]

Bio-Checking: 100%|██████████| 1121/1121 [13:52<00:00,  1.35it/s]


🔬 KẾT QUẢ KIỂM TRA SINH HỌC (Time: 832.88s | 280,122,245 mẫu)
1. TỪ VỰNG DNA (Chỉ A,C,G,T,N): ✅ Sạch
2. SEQUENCE LENGTH:            ✅ 100% chuẩn (601bp)
3. MOTIF (GT/GC cho Donor, AG cho Acc): ✅ Chuẩn xác

4. PHÂN BỐ LABEL (Splicing_types):
   - Label '1': 241,995 mẫu
   - Label '2': 237,072 mẫu
   - Label '0': 279,643,178 mẫu

5. CHROMOSOME COVERAGE:
   - Tìm thấy 22 nhiễm sắc thể.
   - Top 5 Chromosomes: chr1: 28,919,427 | chr2: 23,061,408 | chr3: 19,272,957 | chr7: 15,947,209 | chr12: 15,460,794
